# HeatCast NCR - next-day maximum heat-index dataset (Metro Manila)

Builds `ph_heat_index_next_day.csv` for the **16 NCR cities**, 2015-01-01 to 2025-12-29, validates it, and writes a
pre-modelling analysis (`ncr_dataset_analysis.md`). **No model is trained here.** Scope is NCR only.

The data are **gridded ERA5 reanalysis estimates for representative city coordinates**, not station observations and not 16
independent weather stations: the 16 coordinates share only a few ERA5 grid cells. The analysis report measures how much.

**API quota.** NCR costs about **4,600 weighted Open-Meteo calls**, which fits in **one day** of the free tier (10,000/day). Every
finished city-year is cached (cell 1 keeps the cache on Google Drive), so if a run is interrupted, just run cell 4 again.

Weather data by [Open-Meteo.com](https://open-meteo.com/) (CC BY 4.0); contains modified Copernicus Climate Change Service (ERA5) information. The free API is for non-commercial use.

In [ ]:
# 1) Where to keep the cache and the outputs. Google Drive is strongly recommended.
import os
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/heatcast_ncr'
else:
    BASE = '/content/heatcast_ncr'
WORK_DIR = f'{BASE}/cache'     # resumable API cache + raw responses + run_log.txt
OUT_DIR  = f'{BASE}/output'    # the five final files
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print('cache :', WORK_DIR); print('output:', OUT_DIR)

In [ ]:
# 2) Get the code (skipped if the scripts are already in the working directory) and dependencies.
import os
REPO_URL = 'https://github.com/keyyum/heatcast-ph.git'
BRANCH   = 'claude/ph-heat-index-dataset-m8b8f5'      # change to 'main' once merged
if not os.path.exists('build_ph_heat_index_dataset.py'):
    if not os.path.exists('heatcast-ph'):
        !git clone --depth 1 -b $BRANCH $REPO_URL heatcast-ph
    %cd heatcast-ph
!pip -q install requests pandas numpy tqdm

In [ ]:
# 3) Optional: commercial Open-Meteo API key (leave empty for the free tier).
import os
os.environ['OPEN_METEO_API_KEY'] = ''   # paste a key here to use the customer endpoint with no client-side budget

In [ ]:
# 4) RUN. Default scope is NCR (16 cities). Safe to run repeatedly: finished chunks are never fetched again.
#    Exit code 2 / "STOPPED: daily budget" just means: run again after the budget resets.
#    The preflight call stops early with a clear message if the model returns no data for a variable.
#    Use --max-cities 3 for a quick smoke test (writes a clearly-labelled SUBSET build).
!python build_ph_heat_index_dataset.py --work-dir "$WORK_DIR" --out-dir "$OUT_DIR"

In [ ]:
# 5) The dataset: class counts and percentages for the target.
import os, pandas as pd
path = f'{OUT_DIR}/ph_heat_index_next_day.csv'
if os.path.exists(path):
    df = pd.read_csv(path, parse_dates=['Date'])
    print(df.shape, df['Date'].min().date(), '->', df['Date'].max().date(), '| cities:', df['City'].nunique())
    vc = df['HeatLevelTomorrow'].value_counts().reindex(['Not Hazardous','Caution','Extreme Caution','Danger','Extreme Danger'], fill_value=0)
    display(pd.DataFrame({'rows': vc, 'percent': (vc / vc.sum() * 100).round(2)}))
else:
    print('Dataset not built yet - run cell 4 again.')

In [ ]:
# 6) The pre-modelling analysis: grid series, elevation effect, class sufficiency, leakage, lags, baselines, recommendations.
import os
from IPython.display import Markdown, display
rp = f'{OUT_DIR}/ncr_dataset_analysis.md'
display(Markdown(open(rp, encoding='utf-8').read())) if os.path.exists(rp) else print('Report not written yet.')

In [ ]:
# 7) Optional: re-validate the saved file at any time (read-only).
!python build_ph_heat_index_dataset.py --validate-only "$OUT_DIR/ph_heat_index_next_day.csv"